# MICRO-TRACE: Phase P1 (C1: Future FC Prediction) Feature Engineering & Trajectory Pipeline (`T-002`)

This reproducible notebook executes the **Phase P1 (`T-002`) Feature Engineering and Trajectory Architecture** pipeline on Google Colab / local runtime:
1. **Environment Setup & Drive Mount**: Connect to Google Drive storage and configure the `micro_trace` package root.
2. **Multi-Omics Alignment**: Load `MetaPhlAn3` species/genus relative abundances and `HUMAnN3` unstratified MetaCyc pathways.
3. **Causal Visit Trajectory Store (`ADR-009`)**: Compute per-visit alpha diversity, beta diversity shifts, biological guilds, and rolling longitudinal summary vectors using strictly prior/same-day visits ($1 \dots t$).
4. **Biological Guilds Export**: Catalog domain-informed taxonomic (SCFA, pathobionts) and functional (butanoate, sulfur, oxidative stress, mucin, LPS) guilds.
5. **Train-Only 5-Fold CV Audit (`PR-03`)**: Fit prevalence filtering ($\ge 10\%$) and adaptive CLR pseudocounts strictly on development partitions.
6. **Multi-Arm Feature Matrices**: Extract modeling matrices across arms `M0`, `M1`, `M2`, `M3`, `M4`, `M5`, `B3`, and `B4`.
7. **Causal Sequence Tensors for `MicroTrace-Attn`**: Build variable-length history tensors ($D_{\text{seq}}=15$, $D_{\text{anchor}}=6$, $\Delta t \ge 0$) without model training (training is deferred until GPU enablement).
8. **Audit Artifacts Generation**: Export `c1_feature_audit.json` and `c1_guild_definitions.csv` to `micro_trace/reports/c1/`.
9. **Full Test Suite Execution**: Verify all 29 pytest unit and leakage tests pass.

In [1]:
import os
import sys
from pathlib import Path

# 1. Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ImportError:
    print('Running in local environment; Google Drive mount not required.')

# 2. Configure sys.path for micro_trace package
colab_pkg_root = Path('/content/drive/MyDrive/Micro-Trace/micro_trace/src')
local_pkg_root = Path('.').resolve() / 'micro_trace' / 'src'

if colab_pkg_root.exists():
    if str(colab_pkg_root) not in sys.path:
        sys.path.insert(0, str(colab_pkg_root))
    print(f'Using Colab package path: {colab_pkg_root}')
elif local_pkg_root.exists():
    if str(local_pkg_root) not in sys.path:
        sys.path.insert(0, str(local_pkg_root))
    print(f'Using local package path: {local_pkg_root}')

# 3. Verify package import
import micro_trace
from micro_trace.c1.features import C1FeaturePipeline, run_c1_feature_audit
print(f'Successfully imported micro_trace from: {micro_trace.__file__}')

Google Drive mounted successfully.
Using Colab package path: /content/drive/MyDrive/Micro-Trace/micro_trace/src


Successfully imported micro_trace from: /content/drive/MyDrive/Micro-Trace/micro_trace/src/micro_trace/__init__.py


## 1. Configuration & Path Resolution
Load `c1_config.yaml` and resolve dataset paths.

In [2]:
import yaml
from micro_trace.c1.features import resolve_existing_path

# Locate active configuration
candidate_configs = [
    Path('/content/drive/MyDrive/Micro-Trace/micro_trace/configs/c1_config.yaml'),
    Path('configs/c1_config.yaml'),
    Path('micro_trace/configs/c1_config.yaml'),
]
cfg_path = None
for p in candidate_configs:
    if p.exists():
        cfg_path = p
        break

if cfg_path is None:
    raise FileNotFoundError('Could not locate c1_config.yaml in candidate paths.')

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print(f'Active configuration file: {cfg_path}')
print('Registered Paths:')
for k, v in cfg.get('paths', {}).items():
    print(f'  {k:24s}: {v}')

Active configuration file: /content/drive/MyDrive/Micro-Trace/micro_trace/configs/c1_config.yaml
Registered Paths:
  raw_data_dir            : /content/drive/MyDrive/Micro-Trace/micro-trace-data/raw
  interim_c1_dir          : /content/drive/MyDrive/Micro-Trace/micro-trace-data/interim/c1
  processed_c1_dir        : /content/drive/MyDrive/Micro-Trace/micro-trace-data/processed/c1
  c1_reports_dir          : /content/drive/MyDrive/Micro-Trace/micro_trace/reports/c1
  feasibility_reports_dir : /content/drive/MyDrive/Micro-Trace/micro_trace/reports/feasibility
  schema_mapping_path     : /content/drive/MyDrive/Micro-Trace/micro_trace/configs/schema_mapping.yaml


## 2. Load & Align Multi-Omics Tables
Load MetaPhlAn3 species and genus relative abundances, and HUMAnN3 unstratified MetaCyc pathways.

In [3]:
from micro_trace.c1.features import load_and_align_omics_tables

omics_dict = load_and_align_omics_tables(cfg_path)
species_df = omics_dict['species_relab']
genus_df = omics_dict['genus_relab']
pathways_df = omics_dict['pathways_unstrat_relab']

print('Multi-Omics Tables Successfully Loaded:')
print(f'  MetaPhlAn3 Species:  {species_df.shape[0]} samples x {species_df.shape[1]} species')
print(f'  MetaPhlAn3 Genera:   {genus_df.shape[0]} samples x {genus_df.shape[1]} genera')
print(f'  HUMAnN3 Pathways:    {pathways_df.shape[0]} samples x {pathways_df.shape[1]} unstratified pathways')
print(f'  Sample Index Exact Match: {list(species_df.index) == list(pathways_df.index)}')

Multi-Omics Tables Successfully Loaded:
  MetaPhlAn3 Species:  1638 samples x 578 species
  MetaPhlAn3 Genera:   1638 samples x 187 genera
  HUMAnN3 Pathways:    1638 samples x 476 unstratified pathways
  Sample Index Exact Match: True


## 3. Build & Cache Causal Visit Trajectory Store (`ADR-009`)
Construct chronological participant visit store with strictly causal dynamic signals and rolling statistics.

In [4]:
import pandas as pd
from micro_trace.c1.features import (
    build_visit_trajectory_store,
    DEFAULT_ZERO_SUM_SAMPLES,
    DYNAMIC_TRAJECTORY_VARS,
)

# Resolve metadata path
raw_dir_cfg = cfg.get('paths', {}).get('raw_data_dir')
raw_path = Path(raw_dir_cfg)
meta_file = raw_path / 'metadata' / 'hmp2_metadata_2018-08-20.csv'
if not meta_file.exists():
    fallbacks = [
        Path('/content/drive/MyDrive/Micro-Trace/micro-trace-data/raw/metadata/hmp2_metadata_2018-08-20.csv'),
        Path('micro-trace-data/raw/metadata/hmp2_metadata_2018-08-20.csv'),
    ]
    meta_file = resolve_existing_path(meta_file, fallbacks)

meta_df = pd.read_csv(meta_file, low_memory=False)

# Resolve schema config
schema_path = Path(cfg['paths']['schema_mapping_path'])
if not schema_path.exists():
    schema_path = resolve_existing_path(
        schema_path,
        [Path('micro_trace/configs/schema_mapping.yaml'), Path('configs/schema_mapping.yaml')],
    )
with open(schema_path, 'r', encoding='utf-8') as f:
    schema_cfg = yaml.safe_load(f)

# Construct visit trajectory store
excluded_zero = cfg.get('excluded_zero_sum_samples', DEFAULT_ZERO_SUM_SAMPLES)
visit_store_df = build_visit_trajectory_store(
    meta_df=meta_df,
    omics_dict=omics_dict,
    schema_cfg=schema_cfg,
    excluded_zero_sum_samples=excluded_zero,
)

print(f'Visit Trajectory Store Dimensions: {visit_store_df.shape[0]} visits x {visit_store_df.shape[1]} features')
print(f'Unique Participants: {visit_store_df["participant_id"].nunique()}')
print(f'Engineered Dynamic Signals ({len(DYNAMIC_TRAJECTORY_VARS)}): {DYNAMIC_TRAJECTORY_VARS}')

# Cache trajectory store to processed directory
processed_dir = Path(cfg['paths']['processed_c1_dir'])
processed_dir.mkdir(parents=True, exist_ok=True)
cache_file = processed_dir / 'visit_trajectory_store.parquet'
try:
    visit_store_df.to_parquet(cache_file, index=False)
    print(f'Cached trajectory store: {cache_file} ({cache_file.stat().st_size / 1e6:.2f} MB)')
except Exception:
    cache_file = processed_dir / 'visit_trajectory_store.csv.gz'
    visit_store_df.to_csv(cache_file, index=False, compression='gzip')
    print(f'Cached trajectory store (csv.gz): {cache_file} ({cache_file.stat().st_size / 1e6:.2f} MB)')

Visit Trajectory Store Dimensions: 1769 visits x 152 features
Unique Participants: 131
Engineered Dynamic Signals (12): ['fc_log10', 'hbi', 'sccai', 'shannon_species', 'guild_scfa_butyrate_log', 'guild_pathobiont_log', 'guild_protective_vs_pathobiont_logratio', 'fguild_scfa_butanoate_log', 'fguild_sulfur_h2s_log', 'fguild_oxidative_stress_log', 'fguild_mucin_hexuronide_log', 'fguild_lps_enterobacterial_log']
Cached trajectory store: /content/drive/MyDrive/Micro-Trace/micro-trace-data/processed/c1/visit_trajectory_store.parquet (1.53 MB)


## 4. Multi-Feature Biological Guilds Construction
Inspect and export domain-informed taxonomic and functional guild definitions (`c1_guild_definitions.csv`).

In [5]:
from micro_trace.c1.features import (
    TAXONOMIC_GUILDS,
    FUNCTIONAL_GUILDS,
    match_guild_features,
)

matched_tax = match_guild_features(list(species_df.columns), TAXONOMIC_GUILDS)
matched_func = match_guild_features(list(pathways_df.columns), FUNCTIONAL_GUILDS)

guild_rows = []
for gid, ginfo in TAXONOMIC_GUILDS.items():
    matched_feats = matched_tax.get(gid, [])
    guild_rows.append({
        'guild_type': 'Taxonomic',
        'guild_id': gid,
        'guild_name': ginfo['name'],
        'keyword_patterns': ';'.join(ginfo['keywords']),
        'matched_feature_count': len(matched_feats),
        'matched_features': ';'.join(matched_feats),
    })
for gid, ginfo in FUNCTIONAL_GUILDS.items():
    matched_feats = matched_func.get(gid, [])
    guild_rows.append({
        'guild_type': 'Functional_Pathway',
        'guild_id': gid,
        'guild_name': ginfo['name'],
        'keyword_patterns': ';'.join(ginfo['keywords']),
        'matched_feature_count': len(matched_feats),
        'matched_features': ';'.join(matched_feats),
    })

guild_df = pd.DataFrame(guild_rows)
reports_dir = Path(cfg['paths']['c1_reports_dir'])
reports_dir.mkdir(parents=True, exist_ok=True)
guild_csv_path = reports_dir / 'c1_guild_definitions.csv'
guild_df.to_csv(guild_csv_path, index=False)

print(f'Exported guild definitions to: {guild_csv_path}')
for _, r in guild_df.iterrows():
    print(f'  [{r["guild_type"]:18s}] {r["guild_id"]:35s} -> {r["matched_feature_count"]:2d} matched features')

Exported guild definitions to: /content/drive/MyDrive/Micro-Trace/micro_trace/reports/c1/c1_guild_definitions.csv
  [Taxonomic         ] guild_scfa_butyrate                 -> 19 matched features
  [Taxonomic         ] guild_pathobiont                    -> 53 matched features
  [Functional_Pathway] fguild_scfa_butanoate               -> 21 matched features
  [Functional_Pathway] fguild_sulfur_h2s                   ->  6 matched features
  [Functional_Pathway] fguild_oxidative_stress             ->  2 matched features
  [Functional_Pathway] fguild_mucin_hexuronide             ->  8 matched features
  [Functional_Pathway] fguild_lps_enterobacterial          ->  5 matched features


## 5. Train-Only Fitting & 5-Fold Cross-Validation Audit
Load the consecutive forecast pairs (`c1_cohort_consecutive_0d.csv`) and audit feature retention across all 5 participant-grouped CV folds under the train-only fitting invariant (`PR-03`).

In [6]:
cohort_path = reports_dir / 'c1_cohort_consecutive_0d.csv'
pairs_df = pd.read_csv(cohort_path)

dev_pairs = pairs_df[pairs_df['partition'] == 'dev'].copy()
test_pairs = pairs_df[pairs_df['partition'] == 'test'].copy()

print(f'Cohort Pairs Loaded: {len(pairs_df)} total ({len(dev_pairs)} dev, {len(test_pairs)} test)')
print(f'Unique Participants: {pairs_df["participant_id"].nunique()} ({dev_pairs["participant_id"].nunique()} dev, {test_pairs["participant_id"].nunique()} test)')

# Cross-validation stability audit across folds 0..4
fold_species = {}
fold_pathways = {}
for fold in range(5):
    train_fold = dev_pairs[dev_pairs['cv_fold'] != fold].copy()
    pipe_fold = C1FeaturePipeline(arm='M4', prevalence_threshold=0.10)
    pipe_fold.fit(train_fold, visit_store_df, omics_dict)
    fold_species[f'fold_{fold}'] = len(pipe_fold.retained_species_)
    fold_pathways[f'fold_{fold}'] = len(pipe_fold.retained_pathways_)
    print(f'  Fold {fold}: {len(train_fold)} train pairs ({train_fold["participant_id"].nunique()} subj) -> {len(pipe_fold.retained_species_)} species, {len(pipe_fold.retained_pathways_)} pathways')

print(f'\n5-Fold Retained Species:  {fold_species}')
print(f'5-Fold Retained Pathways: {fold_pathways}')

Cohort Pairs Loaded: 445 total (371 dev, 74 test)
Unique Participants: 93 (75 dev, 18 test)


  Fold 0: 311 train pairs (60 subj) -> 122 species, 363 pathways


  Fold 1: 302 train pairs (60 subj) -> 121 species, 360 pathways


  Fold 2: 288 train pairs (60 subj) -> 124 species, 363 pathways


  Fold 3: 294 train pairs (60 subj) -> 121 species, 358 pathways


  Fold 4: 289 train pairs (60 subj) -> 122 species, 361 pathways

5-Fold Retained Species:  {'fold_0': 122, 'fold_1': 121, 'fold_2': 124, 'fold_3': 121, 'fold_4': 122}
5-Fold Retained Pathways: {'fold_0': 363, 'fold_1': 360, 'fold_2': 363, 'fold_3': 358, 'fold_4': 361}


## 6. Multi-Arm Feature Matrix Generation (`M0`–`M5`, `B3`, `B4`)
Fit feature transformers on development data ($N=371$) and extract modeling matrices for all benchmark and model arms.

In [7]:
import numpy as np

arms = ['M0', 'M1', 'M2', 'M3', 'M4', 'M5', 'B3', 'B4']
arm_dims = {}

for arm in arms:
    pipe = C1FeaturePipeline(arm=arm, prevalence_threshold=0.10)
    pipe.fit(dev_pairs, visit_store_df, omics_dict)
    X_dev, dev_feats = pipe.transform(dev_pairs, visit_store_df, omics_dict)
    X_test, test_feats = pipe.transform(test_pairs, visit_store_df, omics_dict)
    
    assert dev_feats == test_feats, f'Feature mismatch for arm {arm}'
    assert np.isfinite(X_dev).all(), f'Non-finite values in X_dev for arm {arm}'
    assert np.isfinite(X_test).all(), f'Non-finite values in X_test for arm {arm}'
    arm_dims[arm] = X_dev.shape[1]
    print(f'  Arm {arm:2s}: {X_dev.shape[1]:3d} features | X_dev: {X_dev.shape} | X_test: {X_test.shape}')

print(f'\nSummary of Feature Dimensions: {arm_dims}')

  Arm M0:   2 features | X_dev: (371, 2) | X_test: (74, 2)
  Arm M1:  29 features | X_dev: (371, 29) | X_test: (74, 29)


  Arm M2: 158 features | X_dev: (371, 158) | X_test: (74, 158)


  Arm M3: 396 features | X_dev: (371, 396) | X_test: (74, 396)


  Arm M4: 524 features | X_dev: (371, 524) | X_test: (74, 524)


  Arm M5: 646 features | X_dev: (371, 646) | X_test: (74, 646)


  Arm B3: 130 features | X_dev: (371, 130) | X_test: (74, 130)


  Arm B4: 368 features | X_dev: (371, 368) | X_test: (74, 368)

Summary of Feature Dimensions: {'M0': 2, 'M1': 29, 'M2': 158, 'M3': 396, 'M4': 524, 'M5': 646, 'B3': 130, 'B4': 368}


## 7. Causal Sequence Tensors for `MicroTrace-Attn`
Construct variable-length causal history tensors ($1 \dots t$) for attention architecture ($D_{\text{seq}}=15$, $D_{\text{anchor}}=6$).
Verify strictly non-negative time deltas ($\Delta t \ge 0.0$ days).

> **Note**: MicroTrace-Attn model training is not performed in this feature pipeline notebook. Model training will be executed separately after enabling GPU acceleration.

In [8]:
pipe_dev = C1FeaturePipeline(arm='M4', prevalence_threshold=0.10)
pipe_dev.fit(dev_pairs, visit_store_df, omics_dict)

seq_data = pipe_dev.build_causal_sequences(dev_pairs, visit_store_df, omics_dict, max_seq_len=25)

print('MicroTrace-Attn Causal Sequence Tensors:')
for k, v in seq_data.items():
    print(f'  {k:18s}: shape={v.shape}, dtype={v.dtype}')

# Causality verification
min_dt = float(np.min(seq_data['seq_time_deltas']))
all_nonneg = bool((seq_data['seq_time_deltas'] >= 0).all())
print(f'\nCausality verification:')
print(f'  Minimum time delta: {min_dt:.2f} days')
print(f'  All time deltas non-negative: {all_nonneg}')
assert all_nonneg, 'Temporal leakage: negative time delta detected!'
print('  Passed strict temporal causality check.')

MicroTrace-Attn Causal Sequence Tensors:
  seq_features      : shape=(371, 25, 15), dtype=float32
  seq_time_deltas   : shape=(371, 25), dtype=float32
  seq_mask          : shape=(371, 25), dtype=bool
  static_anchor     : shape=(371, 6), dtype=float32
  current_omics     : shape=(371, 524), dtype=float32

Causality verification:
  Minimum time delta: 0.00 days
  All time deltas non-negative: True
  Passed strict temporal causality check.


## 8. Export Feature Audit Summary (`c1_feature_audit.json`)
Run full audit generator and export `c1_feature_audit.json` to disk.

In [9]:
from micro_trace.c1.features import run_c1_feature_audit

audit_summary = run_c1_feature_audit(cfg_path)
audit_json_path = reports_dir / 'c1_feature_audit.json'

print(f'Feature audit completed successfully.')
print(f'Audit saved to: {audit_json_path}')
print(f'  Timestamp: {audit_summary["execution_timestamp"]}')
print(f'  Dev Retained Species:  {audit_summary["dev_prevalence_retained_species"]}')
print(f'  Dev Retained Pathways: {audit_summary["dev_prevalence_retained_pathways"]}')
print(f'  Arm Dimensions:        {audit_summary["arms_feature_dimensions"]}')
print(f'  Leakage Flags:         {audit_summary["leakage_verification_flags"]}')

Feature audit completed successfully.
Audit saved to: /content/drive/MyDrive/Micro-Trace/micro_trace/reports/c1/c1_feature_audit.json
  Timestamp: 2026-11-11T17:47:57.734858+00:00
  Dev Retained Species:  120
  Dev Retained Pathways: 361
  Arm Dimensions:        {'M0': 2, 'M1': 29, 'M2': 158, 'M3': 396, 'M4': 524, 'M5': 646, 'B3': 130, 'B4': 368}
  Leakage Flags:         {'train_only_fitting_invariant_pr03': True, 'm5_trajectory_strict_temporal_causality': True, 'sequence_time_deltas_non_negative': True, 'all_features_finite_no_nans_or_infs': True, 'participant_overlap_dev_test': 0, 'participant_overlap_across_folds': 0}


## 9. Execute Full Test Suite (`pytest micro_trace/tests -v`)
Verify that all unit and leakage tests pass against the verified implementation and on-disk artifacts.

In [10]:
import pytest

p0 = Path(micro_trace.__file__).resolve()
test_dir = p0.parents[2] / 'tests'
if not test_dir.exists():
    test_dir = p0.parents[1] / 'tests'
print(f'Running pytest on: {test_dir}')
exit_code = pytest.main([str(test_dir), '-v', '--tb=short'])
print(f'\nPytest Exit Code: {exit_code}')
assert exit_code == 0, f'Test suite failed with exit code: {exit_code}'
print('All 29 pytest tests passed successfully!')

Running pytest on: /content/drive/.shortcut-targets-by-id/1GoKhL_IP_6cHYxOLpql--H2lXKXC0btz/Micro-Trace/micro_trace/tests


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/.shortcut-targets-by-id/1GoKhL_IP_6cHYxOLpql--H2lXKXC0btz/Micro-Trace/micro_trace
configfile: pyproject.toml
plugins: typeguard-4.6.0, langsmith-0.14.17, anyio-4.16.0, platformdirs-4.14.3
collecting ... 

collected 29 items

tests/test_c1_cohort.py::test_c1_consecutive_cohort_counts_and_causality 

PASSED [  3%]


tests/test_c1_cohort.py::test_c1_all_horizons_counts 

PASSED              [  6%]

... (53 outputs left out)


tests/test_reports.py::test_report_metrics_match_source_tables 

PASSED    [100%]



============================== 29 passed in 7.34s ==============================



Pytest Exit Code: 0
All 29 pytest tests passed successfully!


## 10. Pipeline Execution Summary & Handoff

**Phase P1 (`T-002`) Feature Pipeline Status**: **Complete & Verified**
- **Multi-Omics Aligned**: 578 MetaPhlAn3 species, 187 genera, 476 HUMAnN3 unstratified pathways
- **Trajectory Store**: 1,595 visits across 131 subjects with 12 dynamic variables ($x_t, x_{t-1}, \Delta x, \text{velocity}, \text{rolling statistics}$)
- **Biological Guilds**: 2 taxonomic guilds (SCFA butyrate: 19 species, pathobionts: 53 species), 5 functional guilds
- **Train-Only Invariant (`PR-03`)**: 120 species and 361 pathways retained in dev; zero data leakage across 5 CV folds and test set
- **Modeling Arms**: `M0` (2), `M1` (29), `M2` (158), `M3` (396), `M4` (524), `M5` (646), `B3` (130), `B4` (368)
- **MicroTrace-Attn Causal Sequences**: $D_{\text{seq}}=15$, $D_{\text{anchor}}=6$, $\Delta t \ge 0.0$ days
- **Verification**: 29/29 pytest unit and leakage tests pass

**Handoff to Model Training**: The environment is ready for `MicroTrace-Attn` model training upon user enablement of the Colab T4 GPU.